# NaturalFuzz — generated rows that look like real ones

Splicing values column-wise out of rows already seen means every value genuinely
occurred in that column — the right formats, plausible magnitudes — while the
*combinations* are new.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("naturalfuzz-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## A campaign that splices

In [ ]:
result = bigasterisk.fuzz(spark).fuzz(
    "SELECT cid, SUM(amount) AS total FROM orders WHERE amount > 100 GROUP BY cid",
    {"orders": orders}, iterations=20, strategy="natural", seed=3)

print(result)
print("branches reached:", sorted(result.covered))

## Check

Splicing over a single table cannot break this query, and it should still reach the branches.

In [ ]:
assert result.failures == [], result.failures
assert len(result.covered) > 0
assert result.coverage > 0.0
print("OK")